# Sideris Fine-Tuning Pipeline (Kaggle 2xT4 / Google Colab T4 — Free)

Train **Sideris Lux** (`Qwen3.5-2B`), **Sideris Aether** (`Qwen3.5-4B`), **Sideris Orion** (`Phi-4-mini-reasoning`), or **Sideris Titan** (`Qwen3.5-9B`) on your verified UC3M Aerospace Engineering datasets:
1. **Stage 1 — SFT (QLoRA 4-bit, Unsloth)** on `train_sft.jsonl` (or `train_sft_enriched.jsonl`)
2. **Stage 2 — DPO Preference Alignment** on `train_dpo.jsonl` (zero algebraic skips, no legacy Phase labels, strict Socratic discipline)
3. **Stage 3 — GRPO Verifiable RL** on `train_grpo.jsonl` (deterministic SymPy + KaTeX + SI + Socratic rewards)
4. **Benchmark & Export** to `GGUF Q4_K_M` (Ollama) + `ONNX q4f16` (WebGPU browser) -> Hugging Face Hub

> **Before running:** Upload `train_sft.jsonl`, `eval_holdout.jsonl`, `train_dpo.jsonl`, `train_grpo.jsonl` (generated locally with `python tools/ai_training/build_dataset.py`) and `rewards.py` into `/content/data/` (Colab) or `/kaggle/input/sideris-data/` (Kaggle).

In [ ]:
# 1. Configuration — pick which Sideris model to train
MODEL_CHOICE = "aether"      # "lux" | "aether" | "orion" | "titan"
HF_USERNAME  = "YOUR_HF_USER" # replace with your Hugging Face username
RUN_DPO      = True
RUN_GRPO     = True
SFT_EPOCHS   = 3
DATA_DIR     = "/kaggle/input/sideris-data" if __import__("os").path.exists("/kaggle/input") else "/content/data"
OUT_DIR      = "/kaggle/working/sideris_out" if __import__("os").path.exists("/kaggle/working") else "/content/sideris_out"

RECIPES = {
    "lux":    {"name": "Sideris-Lux",    "base": "Qwen/Qwen3.5-2B",                "r": 16, "lr": 2e-4},
    "aether": {"name": "Sideris-Aether", "base": "Qwen/Qwen3.5-4B",                "r": 32, "lr": 2e-4},
    "orion":  {"name": "Sideris-Orion",  "base": "microsoft/Phi-4-mini-reasoning", "r": 32, "lr": 1.5e-4},
    "titan":  {"name": "Sideris-Titan",  "base": "Qwen/Qwen3.5-9B",                "r": 32, "lr": 1.5e-4},
}
RECIPE = RECIPES[MODEL_CHOICE]
print("Training:", RECIPE)

In [ ]:
# 2. Install Unsloth + TRL + SymPy + ONNX tools
!pip install -q unsloth trl datasets sympy huggingface_hub optimum[onnxruntime]

In [ ]:
# 3. Stage 1 — Supervised Fine-Tuning (QLoRA 4-bit)
import os, pathlib
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel

sft_file = os.path.join(DATA_DIR, "train_sft_enriched.jsonl")
if not os.path.exists(sft_file):
    sft_file = os.path.join(DATA_DIR, "train_sft.jsonl")
print("Using SFT dataset:", sft_file)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=RECIPE["base"],
    max_seq_length=4096,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=RECIPE["r"],
    lora_alpha=RECIPE["r"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

ds = load_dataset("json", data_files=sft_file, split="train")
ds = ds.map(
    lambda b: {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in b["messages"]]},
    batched=True,
)

sft_out = os.path.join(OUT_DIR, f"{RECIPE['name']}-sft")
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    args=SFTConfig(
        output_dir=sft_out,
        dataset_text_field="text",
        max_seq_length=4096,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=SFT_EPOCHS,
        learning_rate=RECIPE["lr"],
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        fp16=True,
        optim="adamw_8bit",
        logging_steps=10,
        save_strategy="epoch",
        seed=42,
    ),
)
trainer.train()
model.save_pretrained(sft_out)
tokenizer.save_pretrained(sft_out)
active_adapter = sft_out
print("Saved SFT adapter:", active_adapter)

In [ ]:
# 4. Stage 2 — DPO Preference Alignment (No algebraic skips, no Phase labels, no Socratic leaks)
dpo_file = os.path.join(DATA_DIR, "train_dpo.jsonl")
if RUN_DPO and os.path.exists(dpo_file):
    from trl import DPOConfig, DPOTrainer
    from unsloth import PatchDPOTrainer
    PatchDPOTrainer()
    dpo_ds = load_dataset("json", data_files=dpo_file, split="train")
    dpo_out = os.path.join(OUT_DIR, f"{RECIPE['name']}-dpo")
    dpo_trainer = DPOTrainer(
        model=model,
        ref_model=None,
        tokenizer=tokenizer,
        train_dataset=dpo_ds,
        args=DPOConfig(
            output_dir=dpo_out,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            num_train_epochs=1,
            learning_rate=5e-6,
            beta=0.1,
            fp16=True,
            optim="adamw_8bit",
            logging_steps=10,
        ),
    )
    dpo_trainer.train()
    model.save_pretrained(dpo_out)
    tokenizer.save_pretrained(dpo_out)
    active_adapter = dpo_out
    print("Saved DPO adapter:", active_adapter)

In [ ]:
# 5. Stage 3 — GRPO Verifiable RL (SymPy + KaTeX + SI Dimensional Check + Socratic Guard)
import sys
grpo_file = os.path.join(DATA_DIR, "train_grpo.jsonl")
if RUN_GRPO and os.path.exists(grpo_file) and os.path.exists(os.path.join(DATA_DIR, "rewards.py")):
    sys.path.insert(0, DATA_DIR)
    from rewards import katex_syntax_reward, no_skip_rigour_reward, socratic_guard_reward, sympy_answer_reward
    from trl import GRPOConfig, GRPOTrainer

    grpo_ds = load_dataset("json", data_files=grpo_file, split="train")
    grpo_out = os.path.join(OUT_DIR, f"{RECIPE['name']}-grpo")
    grpo_trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[katex_syntax_reward, no_skip_rigour_reward, socratic_guard_reward, sympy_answer_reward],
        train_dataset=grpo_ds,
        args=GRPOConfig(
            output_dir=grpo_out,
            learning_rate=5e-6,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
            num_generations=4,
            max_prompt_length=1536,
            max_completion_length=1024,
            num_train_epochs=1,
            fp16=True,
            optim="adamw_8bit",
            logging_steps=5,
        ),
    )
    grpo_trainer.train()
    model.save_pretrained(grpo_out)
    tokenizer.save_pretrained(grpo_out)
    active_adapter = grpo_out
    print("Saved GRPO adapter:", active_adapter)

In [ ]:
# 6. Quick Qualitative & Gate Evaluation on eval_holdout.jsonl
import json
FastLanguageModel.for_inference(model)
eval_file = os.path.join(DATA_DIR, "eval_holdout.jsonl")
if os.path.exists(eval_file):
    samples = [json.loads(l) for l in open(eval_file, encoding="utf-8") if l.strip()][:5]
    for s in samples:
        prompt = tokenizer.apply_chat_template(s["messages"][:-1], tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.3)
        reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        print("===", s["id"], s["task"], "===")
        print(reply[:600], "\n")

In [ ]:
# 7. Export to GGUF (Ollama) + Merged 16-bit (for ONNX q4f16 WebGPU) & Upload to Hugging Face
merged_dir = os.path.join(OUT_DIR, f"{RECIPE['name']}-merged-16bit")
gguf_dir   = os.path.join(OUT_DIR, f"{RECIPE['name']}-gguf")
model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")
model.save_pretrained_gguf(gguf_dir, tokenizer, quantization_method="q4_k_m")

print("Ready! To export ONNX for browser WebGPU:")
print(f"  !optimum-cli export onnx --model {merged_dir} --task text-generation-with-past {OUT_DIR}/{RECIPE['name']}-onnx")
print("To upload to Hugging Face (run huggingface_hub.login() first):")
print(f"  !huggingface-cli upload {HF_USERNAME}/{RECIPE['name']}-GGUF {gguf_dir}")
print(f"  !huggingface-cli upload {HF_USERNAME}/{RECIPE['name']}-ONNX {OUT_DIR}/{RECIPE['name']}-onnx")